In [1]:
import math, json

In [2]:
nodes_dict = json.load(open("../data/nodes.json"))
edges_dict = json.load(open("../data/edges.json"))

# Lengths

In [3]:
def compute_edge_lengths(nodes_dict, edges_dict):
    """
    Compute Euclidean length in km for each edge.
      nodes_dict: {node_id: {"x": float(km), "y": float(km)}}
      edges_dict: {edge_id: {"from": str, "to": str, "directed": bool}}
    Returns:
      {edge_id: float}
    """
    lengths = {}
    for eid, e in edges_dict.items():
        u, v = e["from"], e["to"]
        x1, y1 = nodes_dict[u]["x"], nodes_dict[u]["y"]
        x2, y2 = nodes_dict[v]["x"], nodes_dict[v]["y"]
        lengths[eid] = math.hypot(x2 - x1, y2 - y1)
    return lengths

In [15]:
lengths = compute_edge_lengths(nodes_dict, edges_dict)
for e, v in edges_dict.items():
    v["length_km"] = lengths[e]
    edges_dict[e] = v
json.dump(edges_dict, open("../data/edges.json", "w"), indent=4)

# Probability

In [20]:
prob_vals = [0.1, 0.9]
probs = {}
for eid in edges_dict:
    probs[eid] = {i: {"p": p} for i, p in enumerate(prob_vals)}

json.dump(probs, open("../data/probs_bin.json", "w"), indent=4)

In [22]:
prob_vals = [0.05, 0.10, 0.85]
probs = {}
for eid in edges_dict:
    probs[eid] = {i: {"p": p} for i, p in enumerate(prob_vals)}

json.dump(probs, open("../data/probs_mult.json", "w"), indent=4)

# Fragility

c.f. Byun, J.-E. Ryu, H. and Straub, D. (2026). Branch-and-bound algorithm for efficient reliability analysis of general coherent systems. **Structural Safety** 118, 102653.

In [7]:
# ---- HAZUS fragility table (Table A3) ----

HAZUS_FRAGILITIES = {
    "HWB2": {
        "R": 1.10,
        "beta": 0.6,
        "edges": {
            "e0027", "e0030", "e0031", "e0034", "e0038", "e0040",
            "e0041", "e0042", "e0043", "e0044", "e0058"
        },
    },
    "HWB4": {
        "R": 1.20,
        "beta": 0.6,
        "edges": {
            "e0011", "e0020", "e0024", "e0028", "e0032", "e0045",
            "e0047", "e0048", "e0050", "e0051", "e0052", "e0053",
            "e0059", "e0061", "e0064", "e0070", "e0074", "e0076",
            "e0078", "e0080", "e0085", "e0086", "e0087", "e0089",
            "e0090", "e0091", "e0123", "e0126"
        },
    },
}

DEFAULT_FRAGILITY = {
    "fragility_type": "HWB11",
    "R": 1.10,
    "beta": 0.6,
}


In [8]:
def add_fragilities_to_edges(edges_path):
    # ---- load ----
    with open(edges_path, "r", encoding="utf-8") as f:
        edges = json.load(f)

    # ---- assign fragilities ----
    for eid, edata in edges.items():

        assigned = False

        for hazus_type, info in HAZUS_FRAGILITIES.items():
            if eid in info["edges"]:
                edata["fragility_type"] = hazus_type
                edata["fragility_mean"] = info["R"]
                edata["fragility_std"] = info["beta"]
                edata["fragility_ln_mean"] = math.log(info["R"])
                assigned = True
                break

        if not assigned:
            edata["fragility_type"] = DEFAULT_FRAGILITY["fragility_type"]
            edata["fragility_mean"] = DEFAULT_FRAGILITY["R"]
            edata["fragility_std"] = DEFAULT_FRAGILITY["beta"]
            edata["fragility_ln_mean"] = math.log(DEFAULT_FRAGILITY["R"])

    # ---- write back ----
    with open(edges_path, "w", encoding="utf-8") as f:
        json.dump(edges, f, indent=2)

    return edges


In [9]:
edges_path = "../data/edges.json"
edges = add_fragilities_to_edges(edges_path)

print(edges["e0030"])

{'from': 'n22', 'to': 'n16', 'directed': False, 'length_km': 7.6467965698062095, 'fragility_type': 'HWB2', 'fragility_mean': 1.1, 'fragility_std': 0.6, 'fragility_ln_mean': 0.09531017980432493}
